# CC200 → biological interpretation (UA-PoE, ESWA)

Fills **Table `tab:bio`** and the network-enrichment paragraph in §6.4.

## Why this notebook exists
CC200 is a **functional** parcellation: its parcels carry **indices, not names**. Turning "region 10" into *"left posterior cingulate"* requires the actual atlas volume — it cannot be done from the indices alone, and guessing would be fabrication. This notebook derives it deterministically from the atlas.

## What it produces
1. Centroid of each consensus parcel in **MNI152** space
2. Modal **AAL** / **Harvard–Oxford** label **+ overlap fraction**
3. Modal **Yeo-7 network** membership
4. **Hypergeometric enrichment** test per network (Holm-corrected)


In [1]:
# Run once. Needs internet (downloads ~100 MB of atlases on first run).
!pip install -q nilearn nibabel numpy scipy


[notice] A new release of pip is available: 23.1.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 0. Network / SSL setup — run this first

You hit certificate errors from two hosts:

| host | error | whose fault |
|---|---|---|
| `cluster_roi.projects.nitrc.org` | hostname mismatch | **their server** |
| `www.gin.cnrs.fr` (AAL) | unable to get local issuer certificate | their chain, or a TLS-intercepting proxy on your network |

Note the second error came from `HTTPSConnectionPool` — that's nilearn's internal **requests** downloader, so a plain `urllib` workaround won't touch it. The cell below patches **both** requests and urllib.

⚠️ **Two warnings.**
1. **Do not** set `REQUESTS_CA_BUNDLE`/`SSL_CERT_FILE` to certifi — that's the usual internet advice and it *breaks* downloads on university networks that use a TLS-intercepting proxy (I verified this; it broke a working setup).
2. `INSECURE_DOWNLOADS = True` disables certificate verification, so a man-in-the-middle can't be detected. Acceptable here because these are **public atlases that the next cells sanity-check** (shape, parcel count). Set it back to `False` afterwards, and don't reuse this kernel for anything sensitive.

Try `INSECURE_DOWNLOADS = False` first — if downloads work, leave it off.

In [2]:
INSECURE_DOWNLOADS = True     # set False once downloads work

import ssl, urllib.request, requests, urllib3

if INSECURE_DOWNLOADS:
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    if not getattr(requests.Session, "_patched_verify", False):
        _orig_request = requests.Session.request
        def _no_verify(self, *a, **k):
            k["verify"] = False
            return _orig_request(self, *a, **k)
        requests.Session.request = _no_verify
        requests.Session._patched_verify = True
    ssl._create_default_https_context = ssl._create_unverified_context
    print("TLS verification DISABLED for requests + urllib (this kernel only).")
    print("Remember: set INSECURE_DOWNLOADS = False and restart when you're done.\n")

# connectivity smoke test
try:
    r = requests.get("https://s3.amazonaws.com", timeout=20)
    print("network reachable, S3 responded:", r.status_code)
except Exception as e:
    print("network problem:", type(e).__name__, e)

TLS verification DISABLED for requests + urllib (this kernel only).
Remember: set INSECURE_DOWNLOADS = False and restart when you're done.

network reachable, S3 responded: 200


In [3]:
import numpy as np
import nibabel as nib
from nilearn import datasets, image
from scipy.stats import hypergeom
print("imports ok")

imports ok


## 1. Configure

In [4]:
# The 10 parcels selected by all three attribution methods (Table 4 / Fig. fig:venn)
CONSENSUS = [10, 56, 62, 12, 113, 16, 70, 55, 88, 125]

# Are these ROI *labels* (1-based) or *column indices* (0-based) in your feature vector?
# ABIDE PCP rois_cc200 column j (0-based) = j-th ROI label in the header.
ROI_INDEX_BASE = 1     # 1 => CONSENSUS are ROI labels ; 0 => column indices

YEO_NAMES = {1: "Visual", 2: "Somatomotor", 3: "Dorsal attention",
             4: "Ventral attention / salience", 5: "Limbic",
             6: "Frontoparietal control", 7: "Default mode"}
print(f"{len(CONSENSUS)} consensus parcels, ROI_INDEX_BASE={ROI_INDEX_BASE}")

10 consensus parcels, ROI_INDEX_BASE=1


## 2. Get the CC200 atlas — the *right* one

**Do not use `nilearn.datasets.fetch_atlas_craddock_2012()` here.** Two reasons:

1. Its NITRC host currently fails with an SSL certificate error (their cert doesn't match the hostname — a server-side problem, nothing you can fix).
2. More importantly, it returns a **4-D multi-level clustering map**, which is *not* the file ABIDE used. Its parcel numbering need not match your `rois_cc200` columns — so it could silently give you the wrong anatomy.

The atlas that **actually generated** the ABIDE `rois_cc200` time series is `cc200_roi_atlas.nii.gz` on the PCP's public S3 bucket. That's the one whose numbering matches your feature vector.

In [5]:
import os, ssl, urllib.request
import numpy as np, nibabel as nib
from nilearn import image

CC200_URL = "https://s3.amazonaws.com/fcp-indi/data/Projects/ABIDE/Resources/cc200_roi_atlas.nii.gz"
CC200_FILE = "cc200_roi_atlas.nii.gz"

def download(url, out):
    if os.path.exists(out):
        print(f"{out} already present ({os.path.getsize(out)/1e6:.1f} MB)"); return out
    try:
        urllib.request.urlretrieve(url, out)
    except Exception as e:
        # Fallback: some of these hosts have broken certificates. This disables
        # verification for THIS download only. Acceptable for a public atlas you
        # then sanity-check; do not reuse this context for anything sensitive.
        print(f"normal download failed ({type(e).__name__}), retrying without SSL verification...")
        ctx = ssl.create_default_context()
        ctx.check_hostname = False
        ctx.verify_mode = ssl.CERT_NONE
        opener = urllib.request.build_opener(urllib.request.HTTPSHandler(context=ctx))
        urllib.request.install_opener(opener)
        urllib.request.urlretrieve(url, out)
        urllib.request.install_opener(urllib.request.build_opener())  # restore
    print(f"downloaded {out} ({os.path.getsize(out)/1e6:.1f} MB)")
    return out

download(CC200_URL, CC200_FILE)

cc_img = image.load_img(CC200_FILE)          # 3-D label image, values = ROI ids
cc_data = np.asarray(cc_img.dataobj).astype(int)
present = sorted(int(v) for v in np.unique(cc_data) if v != 0)

print("shape:", cc_img.shape, "| dims:", len(cc_img.shape))
print("parcels found:", len(present), "| id range:", min(present), "-", max(present))
assert len(cc_img.shape) == 3, "Expected a 3-D label map — wrong file?"
assert 150 < len(present) <= 210, f"Expected ~200 parcels, got {len(present)}"
print("\nOK — this is the atlas ABIDE used for rois_cc200.")

cc200_roi_atlas.nii.gz already present (0.0 MB)
shape: (63, 75, 61) | dims: 3
parcels found: 200 | id range: 1 - 200

OK — this is the atlas ABIDE used for rois_cc200.


### 2b. Settle `ROI_INDEX_BASE` definitively

You set `ROI_INDEX_BASE = 1`. Rather than trust that, read it off one of your **actual `.1D` files** — its header lists the ROI ids per column, which is the ground truth for your feature vector.

Point `ONE_1D` at any `*_rois_cc200.1D` you downloaded. If you no longer have one, skip this cell — but then the anatomy rests on an assumption, and you should say so.

In [6]:
ONE_1D = ""   # e.g. "NYU_0051055_rois_cc200.1D" — leave "" to skip

if ONE_1D and os.path.exists(ONE_1D):
    with open(ONE_1D) as f:
        header = [l for l in f if l.startswith("#")][:5]
    print("".join(header))
    # AFNI 3dROIstats headers name columns like  Mean_1  Mean_2 ... => ROI ids
    import re
    ids = [int(m) for m in re.findall(r"Mean_(\d+)", "".join(header))]
    if ids:
        print(f"columns: {len(ids)} | first ids: {ids[:8]} | last: {ids[-4:]}")
        print(f"\nColumn j (0-based) -> ROI id {ids[0]}+j" if ids == list(range(ids[0], ids[0]+len(ids)))
              else "\n!! ROI ids are NOT contiguous — map columns via this list, not by arithmetic")
        missing = sorted(set(present) - set(ids))
        if missing:
            print(f"!! atlas has {len(missing)} parcels absent from the .1D: {missing[:10]}")
        print(f"\n=> your CONSENSUS values should be interpreted as ROI ids "
              f"(ROI_INDEX_BASE=1) if they came from these column names,\n"
              f"   or as 0-based column positions (ROI_INDEX_BASE=0) if they came from "
              f"numpy indexing of the loaded array.")
else:
    print("Skipped — ROI ordering is being ASSUMED, not verified.")
    print("If your attribution indices came from numpy columns (0-based), set ROI_INDEX_BASE = 0.")

Skipped — ROI ordering is being ASSUMED, not verified.
If your attribution indices came from numpy columns (0-based), set ROI_INDEX_BASE = 0.


## 3. Map each consensus parcel

`Overlap` is the fraction of the parcel's voxels carrying the modal label. **Low overlap = the parcel straddles structures** and must be reported cautiously — this is why the paper's table has an overlap column.

**Two version traps handled in the next cell:**

1. **Yeo API changed.** nilearn ≥0.12/0.13 returns `{"maps", "labels"}`; older versions returned `thick_7`. That's your `KeyError`. The cell tries the new API and falls back automatically.
2. **AAL version ≠ your citation.** nilearn ≥0.13 defaults to `version="3v2"` — that's **AAL3** (Rolls et al. 2020). Your paper cites **Tzourio-Mazoyer 2002 = AAL1 = `version="SPM12"`**. The cell pins `SPM12` so the atlas matches the citation. If you'd rather use AAL3, you must change the citation in the paper.

*(Don't pass `url=` to `fetch_atlas_aal` — nilearn only defines its internal filename list when `url is None`, so passing a URL raises an unrelated error. Its own SSL→OSF fallback handles a broken CNRS host.)*

In [7]:
from nilearn import datasets
from scipy.stats import hypergeom

import nilearn
print("nilearn version:", nilearn.__version__)

# ---------------- Harvard-Oxford (API stable) ----------------
ho = datasets.fetch_atlas_harvard_oxford("cort-maxprob-thr25-2mm")
ho_r = image.resample_to_img(ho["maps"], cc_img, interpolation="nearest")
print("Harvard-Oxford ready")

# ---------------- Yeo 7-network ----------------
# nilearn >=0.12/0.13 returns {"maps", "labels"}; older versions returned "thick_7".
try:
    yeo = datasets.fetch_atlas_yeo_2011(n_networks=7, thickness="thick")   # new API
    yeo_maps = yeo["maps"]
except TypeError:
    yeo = datasets.fetch_atlas_yeo_2011()                                   # old API
    yeo_maps = yeo["thick_7"]
except KeyError:
    yeo = datasets.fetch_atlas_yeo_2011()
    yeo_maps = yeo["thick_7"]

yeo_img = image.load_img(yeo_maps)
if len(yeo_img.shape) == 4:              # old files are 4-D with a single volume
    yeo_img = image.index_img(yeo_img, 0)
yeo_r = image.resample_to_img(yeo_img, cc_img, interpolation="nearest")
_vals = sorted(int(v) for v in np.unique(np.asarray(yeo_r.dataobj)) if v != 0)
print(f"Yeo ready | network ids present: {_vals}")
assert set(_vals) <= set(range(1, 8)), "Expected Yeo-7 ids 1..7 — wrong map?"

# ---------------- AAL (optional) ----------------
# IMPORTANT: nilearn >=0.13 defaults to version="3v2" (AAL3, Rolls et al. 2020).
# The paper cites Tzourio-Mazoyer et al. 2002 = AAL1 = version "SPM12".
# Keep SPM12 so the atlas matches the citation.
AAL_VERSION = "SPM12"
USE_AAL = True
aal = aal_r = None
try:
    aal = datasets.fetch_atlas_aal(version=AAL_VERSION)   # do NOT pass url= (nilearn bug)
    aal_r = image.resample_to_img(aal["maps"], cc_img, interpolation="nearest")
    print(f"AAL ready (version {AAL_VERSION} = AAL1, matches Tzourio-Mazoyer 2002)")
except Exception as e:
    print(f"AAL unavailable ({type(e).__name__}) -> continuing with Harvard-Oxford only.")
    print("   Then update Section 6.4: labels come from Harvard-Oxford; drop the AAL mention")
    print("   and the tzourio2002 citation for labelling.")
    USE_AAL = False
print()

def modal_label(mask, atlas_img, labels, indices=None):
    a = np.asarray(atlas_img.dataobj)
    vals = a[mask]; vals = vals[vals != 0]
    if vals.size == 0:
        return "unlabelled", 0.0
    uniq, cnt = np.unique(vals, return_counts=True)
    best = int(uniq[np.argmax(cnt)]); frac = float(cnt.max() / mask.sum())
    if indices is not None:
        try:    name = labels[list(indices).index(best)]
        except (ValueError, IndexError): name = f"idx{best}"
    else:
        name = labels[best] if best < len(labels) else f"idx{best}"
    return (name.decode() if isinstance(name, bytes) else str(name)), frac

# network of EVERY parcel (needed for the enrichment null)
yeo_data = np.asarray(yeo_r.dataobj)
all_net = {}
for p in present:
    v = yeo_data[cc_data == p]; v = v[v != 0]
    all_net[p] = int(np.bincount(v.astype(int)).argmax()) if v.size else 0

rows = []
for rank, idx in enumerate(CONSENSUS, start=1):
    lab = idx if ROI_INDEX_BASE == 1 else present[idx]
    if lab not in present:
        print(f"!! parcel {lab} not in atlas — check ROI_INDEX_BASE"); continue
    m = cc_data == lab
    mni = nib.affines.apply_affine(cc_img.affine, np.argwhere(m).mean(0))
    ho_name,  ho_f  = modal_label(m, ho_r, ho["labels"])
    if USE_AAL:
        aal_name, aal_f = modal_label(m, aal_r, aal["labels"], aal["indices"])
    else:
        aal_name, aal_f = ho_name, ho_f      # fall back to HO as the label
    nid = all_net[lab]; nname = YEO_NAMES.get(nid, "unassigned")
    v = yeo_data[m]; v = v[v != 0]
    net_f = float((v == nid).sum() / m.sum()) if v.size else 0.0
    rows.append(dict(cc=lab, aal=aal_name, aal_f=aal_f, ho=ho_name, ho_f=ho_f,
                     mni=mni, net=nname, net_f=net_f, rank=rank))
    print(f"CC {lab:3d} | {aal_name:26s} ({aal_f:4.0%}) | {nname:28s} ({net_f:4.0%}) "
          f"| MNI ({mni[0]:+4.0f},{mni[1]:+4.0f},{mni[2]:+4.0f})")

nilearn version: 0.13.1
[fetch_atlas_harvard_oxford] Dataset found in C:\Users\AUC\nilearn_data\fsl
Harvard-Oxford ready
[fetch_atlas_yeo_2011] Dataset found in C:\Users\AUC\nilearn_data\yeo_2011
Yeo ready | network ids present: [1, 2, 3, 4, 5, 6, 7]
[fetch_atlas_aal] Dataset created in C:\Users\AUC\nilearn_data\aal_SPM12
[fetch_atlas_aal] Downloading data from https://www.gin.cnrs.fr/AAL_files/aal_for_SPM12.tar.gz ...
[fetch_atlas_aal] Error while fetching file aal_for_SPM12.tar.gz; dataset fetching aborted.
AAL unavailable (SSLError) -> continuing with Harvard-Oxford only.
   Then update Section 6.4: labels come from Harvard-Oxford; drop the AAL mention
   and the tzourio2002 citation for labelling.

CC  10 | Temporal Occipital Fusiform Cortex (  3%) | Visual                       (  1%) | MNI ( +43, -55, -32)
CC  56 | Supramarginal Gyrus, posterior division ( 47%) | Default mode                 ( 46%) | MNI ( -52, -50, +42)
CC  62 | Temporal Fusiform Cortex, posterior division ( 31%

## 4. Network enrichment

Are the 10 parcels concentrated in particular networks, or spread as chance would predict? One-sided hypergeometric test per network, Holm-corrected across the 7.

**A null result is a real result** — a distributed, non-enriched map supports the paper's "network-level disorder, not a focal lesion" reading. Report it either way.

In [8]:
N, n = len(present), len(rows)
raw = []
for nid, nname in YEO_NAMES.items():
    K = sum(1 for p in present if all_net[p] == nid)
    k = sum(1 for r in rows if r["net"] == nname)
    p = hypergeom.sf(k - 1, N, K, n) if k > 0 else 1.0
    raw.append((nname, k, K * n / N, p))

ps = np.array([r[3] for r in raw]); order = np.argsort(ps)
holm = np.empty_like(ps); prev = 0.0
for rank_, i in enumerate(order):
    prev = max(prev, min((len(ps) - rank_) * ps[i], 1.0)); holm[i] = prev

print(f"{'network':30s} {'obs':>4s} {'exp':>6s} {'p':>8s} {'Holm':>8s}")
print("-" * 60)
for (nname, k, exp, p), pc in zip(raw, holm):
    print(f"{nname:30s} {k:4d} {exp:6.1f} {p:8.3f} {pc:8.3f}{'  *' if pc < 0.05 else ''}")

network                         obs    exp        p     Holm
------------------------------------------------------------
Visual                            3    2.1    0.352    1.000
Somatomotor                       1    1.1    0.679    1.000
Dorsal attention                  1    1.0    0.660    1.000
Ventral attention / salience      2    1.1    0.303    1.000
Limbic                            0    0.8    1.000    1.000
Frontoparietal control            2    1.1    0.283    1.000
Default mode                      1    2.1    0.917    1.000
